# Outpatient completion review

Read the **newest** S1_03 output run only after its manifest marks all four validation draws complete. Preserve the S1_03 episode membership and treatment definition. The approved 14-state primary sample excludes Indiana and Texas; the earlier 16-state run remains a sensitivity. The primary descriptive endpoint is recorded completion (`REASON=1`); transfer (`REASON=4`) is reported separately, and a sensitivity counts completion or transfer (`REASON` in 1,4). These are different outcomes, and transfer is not assumed to be treatment success.

This notebook reports arm-specific unweighted and overlap-weighted rates overall, by discharge year, by state, by state–year, and when each state is omitted. Scores and weights come from each saved imputation validation draw; subgroup and omitted-state rows retain full-sample scores. Draw differences and state variation are descriptive sensitivity summaries.
The supplementary coding review reports every REASON category by state, year, and arm, and omits Illinois, Missouri, and both jointly in prespecified descriptive sensitivities. No mapping change is inferred from rate changes.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib,json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'S1/helper').is_dir())
HOME=ROOT/'S1'
folders=sorted((HOME/'outputs').glob('S1_03_outpatient_data_preparation_*'))
if not folders:raise FileNotFoundError('Run S1_03 before S1_04')
source=folders[-1]
manifest_path=source/'manifest.json'
if not manifest_path.is_file():raise FileNotFoundError(f'Newest S1_03 run has no manifest: {source.name}')
source_manifest=json.loads(manifest_path.read_text())
if source_manifest['status'] not in {'candidate_membership_and_four_imputation_draws_verified','candidate_membership_and_four_imputation_draws_verified_overlap_review_required'}:
    raise ValueError(f'Newest S1_03 run is incomplete: {source.name}')
assert source_manifest['imputation']['chains_per_seed']==2
assert source_manifest['imputation']['iterations_per_chain']==10
completion_draws=sorted((x for x in source_manifest['imputation']['draws'] if x['outcome']=='completion'),key=lambda x:x['chain'])
assert len(completion_draws)==2 and [x['chain'] for x in completion_draws]==[1,2]
expected=next(x for x in source_manifest['samples'] if x['outcome']=='completion')
eligible=pd.read_csv(source/'completion_final_eligible.csv.gz',usecols=['episode_source_id','DISYR','STFIPS','treatment','outcome','REASON'])
assert len(eligible)==expected['n'] and eligible.episode_source_id.is_unique
membership=hashlib.sha256(('\n'.join(sorted(eligible.episode_source_id))+'\n').encode()).hexdigest()
assert membership==expected['membership_sha256']
assert eligible.outcome.eq(eligible.REASON.eq(1).astype(int)).all()
OUT=HOME/'outputs'/f'S1_04_outpatient_outcome_review_{datetime.now().astimezone():%Y%m%d_%H%M%S_%f}'
OUT.mkdir()
print('Newest completed source:',source.name)
print('Fixed completion episodes:',len(eligible),'membership SHA-256:',membership)


Newest completed source: S1_03_outpatient_data_preparation_20260926_180001_385313
Fixed completion episodes: 386282 membership SHA-256: de0dfe5d0581975710dc6993a8d70b4b0da94fc02f52b9aab679446628d57c0d


## Outcome rates and geographic sensitivity

Each treatment arm is normalized separately when calculating a weighted outcome rate. A state-specific difference refers to that state's overlap-weighted episodes using scores fitted on the complete outcome sample. The leave-one-state-out rows change the population being described; they are not a refitted causal analysis. Transfer (`REASON=4`) is reported as its own endpoint. `REASON=4` is transfer to another program or facility, so the alternative endpoint is a coding sensitivity.
Illinois (17) and Missouri (29) were flagged by the 2022 crosswalk review before these sensitivities. Year and omitted-state analyses reuse the full-sample propensity scores and keep the frozen main sample unchanged.

In [2]:
def row(frame, outcome, chain, endpoint, analysis, *, state=None, year=None, excluded_state=None, excluded_states=None):
    assert set(frame.treatment)=={0,1}
    assert endpoint in {'completed','transferred','completed_or_transferred'}
    y=frame.REASON.eq(1) if endpoint=='completed' else (frame.REASON.eq(4) if endpoint=='transferred' else frame.REASON.isin([1,4]))
    result=dict(outcome=outcome,chain=chain,endpoint=endpoint,analysis=analysis,
                state=state,year=year,excluded_state=excluded_state,excluded_states=excluded_states,n=len(frame))
    for arm,label in [(0,'nonintensive'),(1,'intensive')]:
        mask=frame.treatment.eq(arm)
        result[f'{label}_n']=int(mask.sum())
        result[f'{label}_unweighted_percent']=100*float(y[mask].mean())
        result[f'{label}_weighted_percent']=100*float(np.average(y[mask],weights=frame.loc[mask,'overlap_weight']))
    result['difference_pp']=result['intensive_weighted_percent']-result['nonintensive_weighted_percent']
    return result
assert row(pd.DataFrame({'treatment':[0,1],'REASON':[2,1],'overlap_weight':[.5,.5]}),
           'completion',1,'completed','check')['difference_pp']==100
assert row(pd.DataFrame({'treatment':[0,1],'REASON':[4,2],'overlap_weight':[.5,.5]}),
           'completion',1,'transferred','check')['difference_pp']==-100
rows=[];concentration=[]
for item in completion_draws:
    path=source/item['file']
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==item['sha256']
    d=pd.read_csv(path,usecols=['episode_source_id','DISYR','STFIPS','treatment','outcome','propensity','overlap_weight'])
    assert len(d)==item['n']==expected['n'] and d.episode_source_id.equals(eligible.episode_source_id)
    for field in ['DISYR','STFIPS','treatment','outcome']:
        np.testing.assert_array_equal(d[field],eligible[field])
    d['REASON']=eligible.REASON.to_numpy()
    np.testing.assert_allclose(d.overlap_weight,np.where(d.treatment.eq(1),1-d.propensity,d.propensity),rtol=0,atol=1e-10)
    chain=item['chain']
    for endpoint in ('completed','transferred','completed_or_transferred'):
        args=('completion',chain,endpoint)
        rows.append(row(d,*args,'overall'))
        for year,g in d.groupby('DISYR'):
            rows.append(row(g,*args,'year',year=int(year)))
        for state,g in d.groupby('STFIPS'):
            rows.append(row(g,*args,'state',state=int(state)))
            for year,h in g.groupby('DISYR'):
                rows.append(row(h,*args,'state_year',state=int(state),year=int(year)))
            rows.append(row(d.loc[d.STFIPS.ne(state)],*args,'leave_one_state_out',excluded_state=int(state),excluded_states=str(int(state))))
        for omitted in ((17,),(29,),(17,29)):
            subset=d.loc[~d.STFIPS.isin(omitted)]
            label=','.join(map(str,omitted))
            if len(omitted)==2:
                rows.append(row(subset,*args,'leave_two_states_out',excluded_states=label))
            for year,g in subset.groupby('DISYR'):
                rows.append(row(g,*args,'year_leave_states_out',year=int(year),excluded_states=label))
    for arm in (0,1):
        sums=d.loc[d.treatment.eq(arm)].groupby('STFIPS').overlap_weight.sum()
        shares=sums/sums.sum()
        concentration.append(dict(chain=chain,arm=arm,largest_state=int(shares.idxmax()),
                                  largest_state_share_percent=100*float(shares.max()),
                                  effective_state_count=float(1/(shares**2).sum())))
review=pd.DataFrame(rows)
n_states=len(source_manifest['states'])
n_years=len(source_manifest['years'])
assert {17,29}.issubset(set(eligible.STFIPS))
assert len(review)==len(completion_draws)*3*(1+n_years+n_states+n_states*n_years+n_states+1+3*n_years)
review.to_csv(OUT/'completion_review.csv',index=False)
reason_profile=eligible.groupby(['STFIPS','DISYR','treatment','REASON']).size().reset_index(name='n')
reason_profile['arm_n']=reason_profile.groupby(['STFIPS','DISYR','treatment']).n.transform('sum')
reason_profile['percent']=100*reason_profile.n/reason_profile.arm_n
assert np.allclose(reason_profile.groupby(['STFIPS','DISYR','treatment']).percent.sum(),100)
print('Illinois and Missouri: completion/transfer rates by year and arm (unweighted):')
display(reason_profile.loc[reason_profile.STFIPS.isin([17,29]) & reason_profile.REASON.isin([1,4]),
       ['STFIPS','DISYR','treatment','REASON','n','arm_n','percent']].round(2))
print('Predefined exclusions: Illinois, Missouri, both; full-sample propensity weights retained:')
display(review.loc[review.analysis.isin(['leave_one_state_out','leave_two_states_out','year_leave_states_out'])
       & review.excluded_states.isin(['17','29','17,29']) & review.endpoint.eq('completed'),
       ['chain','analysis','year','excluded_states','n','difference_pp']].round(3))
summary=review.loc[review.analysis.eq('overall'),['chain','endpoint','nonintensive_weighted_percent','intensive_weighted_percent','difference_pp']]
state=review.loc[review.analysis.eq('state') & review.endpoint.eq('completed')]
loo=review.loc[review.analysis.eq('leave_one_state_out') & review.endpoint.eq('completed')]
print('Weighted overall contrasts (descriptive percentages and percentage points):')
display(summary.round(3))
print('Recorded-completion state range by draw:')
display(state.groupby('chain').difference_pp.agg(['min','max']).round(3))
print('Recorded-completion leave-one-state-out range by draw:')
display(loo.groupby('chain').difference_pp.agg(['min','max']).round(3))
print('Recorded-completion differences by year:')
display(review.loc[review.analysis.eq('year') & review.endpoint.eq('completed'),
                   ['chain','year','n','difference_pp']].round(3))
print('Geographic weight concentration:')
display(pd.DataFrame(concentration).round(3))
output_manifest=dict(status='descriptive_review_complete_inference_pending',source_run=source.name,
                     source_manifest_sha256=hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
                     completion_membership_sha256=membership,
                     reviewed_draws=[{'file':x['file'],'sha256':x['sha256']} for x in completion_draws],
                     output_file='completion_review.csv',output_sha256=hashlib.sha256((OUT/'completion_review.csv').read_bytes()).hexdigest(),
                     role='descriptive outcome and geographic checks; no uncertainty interval or causal claim',
                     reason_profile_state_year_arm=reason_profile.to_dict('records'),
                     predefined_coding_sensitivity_states=[17,29],
                     coding_sensitivity_note='2022 crosswalk flags motivate these checks; 2023 mapping and causes of observed changes remain unknown')
(OUT/'manifest.json').write_text(json.dumps(output_manifest,indent=2)+'\n')
print('Output directory:',OUT)


Illinois and Missouri: completion/transfer rates by year and arm (unweighted):
Predefined exclusions: Illinois, Missouri, both; full-sample propensity weights retained:
Weighted overall contrasts (descriptive percentages and percentage points):
Recorded-completion state range by draw:
Recorded-completion leave-one-state-out range by draw:
Recorded-completion differences by year:
Geographic weight concentration:
Output directory: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs/S1_04_outpatient_outcome_review_20260926_193521_946088


,STFIPS,DISYR,treatment,REASON,n,arm_n,percent
164,17,2022,0,1,2053,5165,39.75
167,17,2022,0,4,593,5165,11.48
171,17,2022,1,1,409,2071,19.75
174,17,2022,1,4,827,2071,39.93
177,17,2023,0,1,4089,13263,30.83
180,17,2023,0,4,2429,13263,18.31
184,17,2023,1,1,873,5180,16.85
187,17,2023,1,4,1993,5180,38.47
191,29,2022,0,1,3615,13748,26.29
194,29,2022,0,4,3687,13748,26.82


,chain,analysis,year,excluded_states,n,difference_pp
30,1,leave_one_state_out,NaN,17,360603,-1.062
34,1,leave_one_state_out,NaN,29,337054,-0.882
59,1,year_leave_states_out,2022.0,17,196296,-0.688
60,1,year_leave_states_out,2023.0,17,164307,-1.507
61,1,year_leave_states_out,2022.0,29,171249,0.590
62,1,year_leave_states_out,2023.0,29,165805,-2.355
63,1,leave_two_states_out,NaN,"17,29",311375,0.540
64,1,year_leave_states_out,2022.0,"17,29",164013,1.624
65,1,year_leave_states_out,2023.0,"17,29",147362,-0.635
228,2,leave_one_state_out,NaN,17,360603,-1.067


,chain,endpoint,nonintensive_weighted_percent,intensive_weighted_percent,difference_pp
0,1,completed,31.218,29.076,-2.142
66,1,transferred,12.093,16.281,4.188
132,1,completed_or_transferred,43.311,45.356,2.046
198,2,completed,31.225,29.079,-2.145
264,2,transferred,12.090,16.281,4.191
330,2,completed_or_transferred,43.315,45.361,2.046


,min,max
chain,,
1,-23.667,38.544
2,-23.634,38.586


,min,max
chain,,
1,-3.148,-0.882
2,-3.150,-0.895


,chain,year,n,difference_pp
1,1,2022.0,203532,-1.427
2,1,2023.0,182750,-2.932
199,2,2022.0,203532,-1.439
200,2,2023.0,182750,-2.924


,chain,arm,largest_state,largest_state_share_percent,effective_state_count
0,1,0,34,28.484,7.015
1,1,1,34,28.481,7.015
2,2,0,34,28.481,7.017
3,2,1,34,28.478,7.017


## Readiness audit: outcome observation and structural covariate gaps

Frequency reduction is observed only when the admission substance is uniquely matched to a discharge substance with known frequency. Compare observed frequency outcome membership with its baseline-user target, by state and treatment. Also flag any adjustment covariate entirely unobserved within a retained state–year cell. These diagnostics do not change S1_03 membership or fill missing outcomes. A state-level all-missing covariate cannot be checked against observed values within that state, even if chained imputation creates values.

In [3]:
fields=json.loads((HOME/'helper/assets/sample_specification.json').read_text())['fields']
full=pd.read_csv(source/'completion_final_eligible.csv.gz',usecols=list(dict.fromkeys(
    ['episode_source_id','STFIPS','DISYR','treatment','REASON','FREQ1']+fields)))
structural=[]
for (state,year),g in full.groupby(['STFIPS','DISYR']):
    for field in fields:
        if g[field].isna().all():
            structural.append(dict(state=int(state),year=int(year),field=field,n=len(g)))
structural=pd.DataFrame(structural)
assert structural.empty if source_manifest.get('analysis_role')=='primary_14_state' else {(r.state,r.field) for r in structural.itertuples()}=={(18,'PRIMINC'),(48,'PSYPROB')}
frequency_info=next(x for x in source_manifest['samples'] if x['outcome']=='frequency')
frequency_ids=pd.read_csv(source/'frequency_final_eligible.csv.gz',usecols=['episode_source_id']).episode_source_id
assert len(frequency_ids)==frequency_info['n'] and frequency_ids.is_unique
frequency_membership=hashlib.sha256(('\n'.join(sorted(frequency_ids))+'\n').encode()).hexdigest()
assert frequency_membership==frequency_info['membership_sha256']
target=full.loc[full.REASON.isin([1,2,3,4,5,7]) & full.FREQ1.isin([2,3]),
                ['episode_source_id','STFIPS','DISYR','treatment']].copy()
target['observed']=target.episode_source_id.isin(set(frequency_ids))
assert int(target.observed.sum())==len(frequency_ids)
frequency_availability=target.groupby(['STFIPS','DISYR','treatment'])['observed'].agg(['sum','count']).reset_index()
frequency_availability['percent']=100*frequency_availability['sum']/frequency_availability['count']
state_availability=target.groupby(['STFIPS','treatment'])['observed'].agg(['sum','count']).reset_index()
state_availability['percent']=100*state_availability['sum']/state_availability['count']
print('State–year adjustment fields with no observed values:')
display(structural)
print('Frequency outcome availability by state and arm:')
display(state_availability.pivot(index='STFIPS',columns='treatment',values='percent').round(1))
print('Frequency target and observed episodes:',len(target),len(frequency_ids))
output_manifest['readiness_audit']={
    'state_year_all_missing_adjustment_fields':structural.to_dict('records'),
    'frequency_target_n':len(target),'frequency_observed_n':len(frequency_ids),
    'frequency_availability_by_state_year_arm':frequency_availability.to_dict('records'),
    'frequency_membership_sha256':frequency_membership,
    'interpretation':'Differential frequency observation remains a design issue; no sample rules changed by this review'
}
output_manifest['status']='descriptive_review_complete_readiness_issues_identified'
(OUT/'manifest.json').write_text(json.dumps(output_manifest,indent=2)+'\n')


State–year adjustment fields with no observed values:
Frequency outcome availability by state and arm:
Frequency target and observed episodes: 220401 152566


""


treatment,0,1
STFIPS,,
1,72.6,77.6
5,98.7,99.7
8,98.5,99.6
9,94.3,94.0
13,91.0,93.6
15,85.3,71.4
17,89.0,93.5
29,59.3,55.4
34,34.2,25.7


## Interpretation gate

The sample files pass integrity checks. Interpretation of outpatient completion requires attention to state reporting practices. The 2022 state crosswalks show different mappings of completion/transfer: Illinois includes a category for completing the current level of care under national transfer; Missouri includes “additional services advised, referral made” under national completion. Crosswalks are descriptive reporting documentation, not proof that the observed state differences are caused by coding. The latest available local crosswalk set is for 2022, so 2023 mapping stability remains unverified. The 14-state primary sample excludes Indiana and Texas, where one adjustment field each is not collected. The earlier 16-state run remains available as a sensitivity. Frequency-outcome observation also differs across states and arms. This review retains the full sample and documents endpoint coding, adjustment, and outcome-observation issues for subsequent inference.
The absence of 2023 crosswalks prevents verification of year-specific mappings. The added state/year/arm REASON profile and Illinois/Missouri exclusions identify sensitivity to coding concerns, but cannot distinguish coding changes from case mix or other reporting changes.